DC Cable Routing - Sensitivity Analysis (Minas do Sol, inverters in the middle)
=
This is a copy of `Tracker DC - Minas`, run on a test drawing (`Análise inversor nos meios.dwg`) where the inverters sit in the middle of the tracker rows, between the tracker ends, instead of at their ends. The same routing produces the cable, duct and trench lengths of this layout, to compare with the base layout.

The comparison itself isn't in the notebook: it only produces this scenario's scripts and totals.

**Differences from `Tracker DC - Minas`:**
- Drawing `Análise inversor nos meios.dwg`, loaded into DuckDB under the project name `PV Minas do Sol-teste`, so the base project `PV Minas do Sol` isn't touched.
- The tracker break threshold is 35 instead of 37.5.
- `distance4` / `distance5` are swapped (0.1825 in X, 0.675 in Y), which suggests the inverter block is rotated 90° compared with the base drawing. The stretch points are closer to the inverter (0.5 / 0.45 / 0.4 / 0.35).
- There is an extra debug export, `teste3333.xlsx`.
- The `.scr` files get a `-teste` suffix and go to `...\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\`.
- **The Excel outputs (`Lengths.xlsx`, `SCRIPTS.xlsx`, `cables.xlsx`, `teste.xlsx`, `teste1.xlsx`, `structure_per_PST.xlsx`) use the same Desktop paths as the base notebook.** Copy or rename the base results before running this one, or the base results are overwritten and there is nothing to compare with.

The inverters sitting between the tracker ends is exactly the `M` ("middle") case of the exit logic, so the notes on `M` below matter most in this notebook.

**Workflow overview:**
1. Convert the DWG to DXF and load it into the `cad_entities` table of `pv_projects.duckdb`, under the project `PV Minas do Sol-teste`.
2. Select the string blocks and the inverter blocks, and build the key `GRP` = `UFV.PST.INV` on both.
3. Split each tracker line into **SubGroups** (consecutive strings of the same inverter) and **Groups** (the physical tracker), and find their north/south ends.
4. For every SubGroup, choose where the trench leaves the tracker (`closest_y`) and the duct arrangement for its string count.
5. Accumulate the string count along the trench toward the inverter (`STR_CUM_TO_INV`).
6. Compute the routes and write the AutoCAD scripts (cables, trenches, ducts, layered trenches), plus Excel files with the lengths.

**Route geometry:**
- Trackers run north-south.
- A cable leaves its string at the exit on the inverter side and runs along the tracker to `closest_y`.
- From there it goes diagonally to an intermediary point, `distance3` to the side of the tracker line at the inverter's Y.
- It then passes four extra "stretch" points close to the inverter, so the end can be stretched in AutoCAD, and ends on the inverter edge.
- The trench and the ducts follow the same path, starting at `closest_y`.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `Análise inversor nos meios.dwg` in `C:\Users\Usuario\Desktop\projetos\Minas`. The name is repeated in three paths in the first code cell. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** strings and inverters are block references (`INSERT`) placed directly in model space.
- **Block names** (case-sensitive):
  - Strings: the name matches `LIKE 'String_Tracker_%'` (e.g. `String_Tracker_1X174p_1`). In `LIKE`, `_` also matches any single character. No other block name may start this way.
  - Inverters: exactly `MDS-LAYOUT$0$PRF_Inversor`.
- **Attributes:**
  - Every string has `UFV`, `PST` and `INV` filled with integers. They are cast with `astype(int)`, so an empty value fails.
  - Every inverter has `UFV`, `PST` and `INV`, written the same way as on its strings. `UFV` and `PST` are compared as text (`1` ≠ `01`); `INV` is zero-padded on both sides.
  - Each `UFV.PST.INV` belongs to exactly one inverter. A duplicate duplicates its strings and trenches.
- **Geometry:**
  - All strings of a tracker have the same X (rounded to 1 m).
  - The string insertion point is the string midpoint, 15.8086 (`distance1`) from each string end.
  - Consecutive strings of one tracker are at most 35 apart in Y. A larger gap starts a new tracker.
  - The tracker lines are 6.8 apart (`pitch`).
  - The inverter insertion point is its centre. The inverter is 0.365 wide (X) and 1.35 high (Y) (`distance4` / `distance5` are half of these).
  - An inverter counts as being on a tracker line only when its X rounds to the same metre as that line.
- **Layers:** the layers of the blocks aren't used by the routing, they are only counted. The scripts switch to `EMF_cabo_solar_positivo`, `EMF_bt_vala`, `EMF_bt_eletroduto_1S` to `_4S` and `EMF_vala_CC_1C` to `EMF_vala_CC_<max>C`. `LAYER S` only sets an existing layer as current, so these layers must already exist in the target drawing.
- **When running the `.scr` in AutoCAD:** turn running object snaps off (`OSMODE = 0`), so the polyline points aren't snapped to nearby geometry.

Converting the DWG and loading it into DuckDB
=
- `dwg_folder` holds the source DWG and `temp_folder` a working copy. The DXF is written back to `dwg_folder`.
- Copies the DWG into `temp_folder` so the ODA File Converter only processes this file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) turns it into a DXF (ACAD2018).
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into the `cad_entities` table of `pv_projects.duckdb` (in the notebook's working folder), under the project name `PV Minas do Sol-teste`.
  - Old rows of that project are deleted first.
  - Block references keep their name, insertion point, rotation, color and attributes (`UFV`, `PST`, `INV`, ...).
- At the end, the DXF and the temporary DWG are deleted. The connection `conn` stays open, and the last cell closes it.

In [4]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\Minas"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "Análise inversor nos meios.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "Análise inversor nos meios.dxf")

temp_dwg_file = os.path.join(temp_folder, "Análise inversor nos meios.dwg")

conn = load_to_duckdb(dxf_file, "PV Minas do Sol-teste")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)


Project PV Minas do Sol-teste already exists
Removed existing data for project: PV Minas do Sol-teste
Successfully loaded project
Loaded 2621 entities for project: PV Minas do Sol-teste
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Loading the project data and counting the blocks
=
- `ELM` is a DuckDB relation with this project's columns used later (`layer`, `name`, `X`, `Y`, `UFV`, `PST`, `INV`, `GRP`, `STR`, `rotation`). The next queries use it by its Python variable name.
- `DESCRIBE ELM` prints the column types. The attributes are text (`VARCHAR`).
- Counts the entities per block name. Use it to check the string and inverter block names selected in the next steps.
- Counts the entities per layer and block name (the number of each structure type per PST layer), and writes that table to `C:\Users\Usuario\Desktop\structure_per_PST.xlsx`.

In [5]:
ELM = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP, STR, rotation FROM cad_entities WHERE project_name='PV Minas do Sol-teste'")

print(ELM)
print(conn.execute("DESCRIBE ELM").fetchdf())

result = conn.execute("""
    SELECT
        name,
        COUNT(*) as count
    FROM ELM
    GROUP BY name
    ORDER BY count DESC
""").fetchdf()
print(result)

structures_per_PST = result = conn.execute("""
    SELECT
        layer,
        name,
        COUNT(*) as count
    FROM ELM
    GROUP BY layer, name
    ORDER BY count DESC
""").fetchdf()
structures_per_PST.to_excel(r"C:\Users\Usuario\Desktop\structure_per_PST.xlsx")

┌───────────────────────┬─────────────────────┬────────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┐
│         layer         │        name         │     X      │     Y     │   UFV   │   PST   │   INV   │   GRP   │   STR   │ rotation  │
│        varchar        │       varchar       │   float    │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │   float   │
├───────────────────────┼─────────────────────┼────────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ G-SITE                │ MDS-LG$0$Portão_ped │  511930.56 │ 8087438.5 │ NULL    │ NULL    │ NULL    │ NULL    │ NULL    │  359.7934 │
│ G-SITE                │ MDS-LG$0$Portão_ped │  514778.47 │ 8087966.0 │ NULL    │ NULL    │ NULL    │ NULL    │ NULL    │ 28.305109 │
│ G-SITE                │ MDS-LG$0$Portão_ped │ 515622.375 │ 8088216.5 │ NULL    │ NULL    │ NULL    │ NULL    │ NULL    │ 23.925589 │
│ G-SITE                │ MDS-LG$0$Portão_ped │  515975

Strings: selection and preparation
=
- Selects the blocks whose name matches `String_Tracker_%`. The match is case-sensitive, and `_` is a one-character wildcard in `LIKE`.
- `GRP` = `UFV.PST.INV` with `INV` zero-padded (e.g. `1.1.07`). It is built from the text attributes **before** the integer cast, so it has the same format as the inverter key.
- Casts `UFV`, `PST` and `INV` to `int`. This fails if any string has an empty attribute.
- `Line` = X rounded to the metre. All strings of the same tracker share it.

In [6]:
strings = conn.sql("SELECT * FROM ELM WHERE name LIKE 'String_Tracker_%'")

strings_df = strings.fetchdf()

strings_df["GRP"] = strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str) + "." + strings_df["INV"].astype(str).str.zfill(2)

strings_df['UFV'] = strings_df['UFV'].astype(int)
strings_df['PST'] = strings_df['PST'].astype(int)
strings_df['INV'] = strings_df['INV'].astype(int)
strings_df['Line'] = round(strings_df['X'],0)
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

                   X             Y          UFV     PST          INV  \
count    1274.000000  1.274000e+03  1274.000000  1274.0  1274.000000   
mean   513187.312500  8.086089e+06     5.728414     1.0    12.020408   
std       162.772186  3.132291e+02     0.709066     0.0     8.561660   
min    512930.531250  8.085642e+06     5.000000     1.0     1.000000   
25%    513048.531250  8.085814e+06     5.000000     1.0     4.000000   
50%    513175.343750  8.085988e+06     6.000000     1.0    12.000000   
75%    513304.531250  8.086418e+06     6.000000     1.0    20.000000   
max    513562.937500  8.086523e+06     7.000000     1.0    27.000000   

          rotation           Line  
count  1274.000000    1274.000000  
mean     86.467819  513187.375000  
std      89.965973     162.767715  
min       0.000000  512931.000000  
25%       0.000000  513049.000000  
50%       0.000000  513175.000000  
75%     180.000000  513305.000000  
max     180.000000  513563.000000  
<class 'pandas.DataFrame'>


Inverters: selection and preparation
=
- Selects the blocks named exactly `MDS-LAYOUT$0$PRF_Inversor`, and renames the insertion point to `inverter_x` / `inverter_y`.
- `GRP` = `UFV.PST.INV`, with `INV` zero-padded to 2 digits. This key links the strings to their inverter.
- `inverter_line` = inverter X rounded to the metre. It is compared later with the strings' `Line` to know whether the inverter sits on the same tracker line.
- The sort (Y descending, X ascending) only affects the printout. The later merges don't depend on it.

In [7]:
inverters = conn.sql("SELECT * FROM ELM WHERE name='MDS-LAYOUT$0$PRF_Inversor'")

inverter_df = inverters.fetchdf()

inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation'], inplace=True)
inverter_df["GRP"] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + "." + inverter_df["INV"].astype(str).str.zfill(2)
inverter_df['inverter_line'] = round(inverter_df['inverter_x'],0)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

          inverter_x    inverter_y  inverter_line
count      54.000000  5.400000e+01      54.000000
mean   513194.031250  8.086118e+06  513194.156250
std       166.211319  3.233938e+02     166.241806
min    512940.718750  8.085728e+06  512941.000000
25%    513051.734375  8.085832e+06  513051.500000
50%    513182.031250  8.086134e+06  513182.000000
75%    513311.273438  8.086436e+06  513311.500000
max    513546.031250  8.086470e+06  513546.000000
<class 'pandas.DataFrame'>
Index: 54 entries, 12 to 53
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   layer          54 non-null     str    
 1   inverter_x     54 non-null     float32
 2   inverter_y     54 non-null     float32
 3   UFV            54 non-null     str    
 4   PST            54 non-null     str    
 5   INV            54 non-null     str    
 6   GRP            54 non-null     str    
 7   STR            54 non-null     str    
 8   inverter_line  54 

Splitting the tracker lines into SubGroups and Groups
=
- Sorts the strings by `Line` ascending, then Y descending, so each tracker line is walked from north to south.
- Compared with the previous string, a new block starts when:
  - the `Line` changes (condition 1),
  - the `GRP` changes, i.e. the string belongs to another inverter (condition 2, SubGroup only),
  - the Y gap is larger than 35 (condition 3: a break in the structure, i.e. another tracker on the same line).
- `SubGroup` = consecutive strings on the same tracker and the same inverter. Its string count sets the ducts.
- `Group` = consecutive strings on the same tracker, whatever their inverter, i.e. the physical tracker. Its ends are where the trenches can leave the tracker.
- The 35 threshold must be larger than the string spacing along a tracker, and smaller than the gap between two trackers. Check it for each project.
- **Here 35 is right at the string spacing:** some consecutive strings in this drawing are exactly 35 apart. A tiny offset in the drawing is enough to split a tracker in two.
- Writes `strings_df` to `C:\Users\Usuario\Desktop\teste.xlsx` for checking.

In [8]:
strings_df.sort_values(by=['Line', 'Y'], ascending=[True, False], inplace=True)


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Different PSTs
condition2 = strings_df["GRP"] != strings_df["GRP"].shift()

#Break in the Structure
condition3 = abs(strings_df["Y"] - strings_df["Y"].shift()) > 35

break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition3

strings_df['SubGroup'] = break_condition_1.cumsum() # The SubGroup represents strings that are on the same Line, are from the same inverter and don't have a break between them. Used to count the number of strings that are going down the pile in the end.
strings_df['Group'] = break_condition_2.cumsum() # The Group represents strings that are on the same Line and don't have a break between them. Used to calculate where the start and end piles are.
print(strings_df)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

       layer                     name             X          Y  UFV  PST  INV  \
573  MDS_7.1  String_Tracker_1X174p_1  512930.53125  8086022.5    7    1    1   
574  MDS_7.1  String_Tracker_1X174p_2  512930.53125  8085988.5    7    1    1   
575  MDS_7.1  String_Tracker_1X174p_3  512930.53125  8085954.5    7    1    1   
578  MDS_5.1  String_Tracker_1X174p_3  512930.53125  8085918.5    5    1    1   
577  MDS_5.1  String_Tracker_1X174p_2  512930.53125  8085884.5    5    1    1   
..       ...                      ...           ...        ...  ...  ...  ...   
101  MDS_6.1  String_Tracker_1X145p_1  513562.93750  8086488.5    6    1   26   
97   MDS_6.1  String_Tracker_1X145p_2  513562.93750  8086454.5    6    1   26   
98   MDS_6.1  String_Tracker_1X145p_3  513562.93750  8086419.5    6    1   26   
99   MDS_6.1  String_Tracker_1X145p_4  513562.93750  8086384.5    6    1   26   
100  MDS_6.1  String_Tracker_1X145p_5  513562.93750  8086350.5    6    1   26   

        GRP STR  rotation  

String ends and tracker ends
=
- `distance1` = 15.8086 is half the string length (string midpoint to the string's end pile). `N exit` / `S exit` are the north/south ends of each string.
- For each SubGroup and each Group, takes the northernmost `N exit` and the southernmost `S exit`. These are the ends of the inverter's stretch and the ends of the whole tracker.
- `cables_df` = strings inner-joined with the inverters on `GRP`: one row per string, with its inverter's coordinates.
  - Strings with no matching inverter are dropped silently.
  - A `GRP` shared by two inverters duplicates its strings.
- `No. of Strings` (the number of strings in the SubGroup) is only added after the merge, so `cables_df` doesn't have it.

In [9]:
distance1 = 15.8086 # Distance between the middle point of the string and the tracker's pile.

strings_df['N exit'] = strings_df['Y'] + distance1
strings_df['S exit'] = strings_df['Y'] - distance1

strings_df['SubGroup - Position Y - N'] = strings_df.groupby("SubGroup")["N exit"].transform('max')
strings_df['SubGroup - Position Y - S'] = strings_df.groupby("SubGroup")["S exit"].transform('min')

strings_df['Group - Position Y - N'] = strings_df.groupby("Group")["N exit"].transform('max')
strings_df['Group - Position Y - S'] = strings_df.groupby("Group")["S exit"].transform('min')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['Line'].transform('count')

print(strings_df)

       layer                     name             X          Y  UFV  PST  INV  \
573  MDS_7.1  String_Tracker_1X174p_1  512930.53125  8086022.5    7    1    1   
574  MDS_7.1  String_Tracker_1X174p_2  512930.53125  8085988.5    7    1    1   
575  MDS_7.1  String_Tracker_1X174p_3  512930.53125  8085954.5    7    1    1   
578  MDS_5.1  String_Tracker_1X174p_3  512930.53125  8085918.5    5    1    1   
577  MDS_5.1  String_Tracker_1X174p_2  512930.53125  8085884.5    5    1    1   
..       ...                      ...           ...        ...  ...  ...  ...   
101  MDS_6.1  String_Tracker_1X145p_1  513562.93750  8086488.5    6    1   26   
97   MDS_6.1  String_Tracker_1X145p_2  513562.93750  8086454.5    6    1   26   
98   MDS_6.1  String_Tracker_1X145p_3  513562.93750  8086419.5    6    1   26   
99   MDS_6.1  String_Tracker_1X145p_4  513562.93750  8086384.5    6    1   26   
100  MDS_6.1  String_Tracker_1X145p_5  513562.93750  8086350.5    6    1   26   

        GRP STR  rotation  

One row per SubGroup
=
Collapses `strings_df` into `groups_df`, with one row per SubGroup. Each row keeps the first string's values, and after the sort that is the northernmost string, so `Y` is the midpoint of the SubGroup's top string.

Renamed columns:
- `Y-N-STR` / `Y-S-STR` = the SubGroup's north/south ends.
- `Y-N-TRK` / `Y-S-TRK` = the tracker's (Group's) north/south ends.

Prints the largest number of strings in a SubGroup. The next step's `duct_map` must cover it.

In [10]:
groups_df = strings_df.groupby("SubGroup")[['GRP','X', 'Y', 'SubGroup - Position Y - N', 'SubGroup - Position Y - S', 'Group - Position Y - N', 'Group - Position Y - S', 'No. of Strings', 'Line']].first()
groups_df.rename(columns={'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 5

             GRP             X          Y    Y-N-STR    Y-S-STR    Y-N-TRK  \
SubGroup                                                                     
1         7.1.01  512930.53125  8086022.5  8086038.5  8085938.5  8086038.5   
2         5.1.01  512930.53125  8085918.5  8085934.5  8085835.0  8085934.5   
3         5.1.01  512930.53125  8085814.0  8085830.0  8085730.0  8085830.0   
4         5.1.20  512930.53125  8085710.0  8085726.0  8085626.0  8085726.0   
5         6.1.01  512932.93750  8086522.5  8086538.5  8086438.5  8086538.5   
...          ...           ...        ...        ...        ...        ...   
453       6.1.26  513542.53125  8086418.5  8086434.5  8086335.0  8086434.5   
454       6.1.26  513549.34375  8086522.5  8086538.5  8086438.5  8086538.5   
455       6.1.26  513549.34375  8086418.5  8086434.5  8086335.0  8086434.5   
456       6.1.26  513556.12500  8086488.5  8086504.5  8086334.5  8086504.5   
457       6.1.26  51356

Duct arrangement, exit side and trench exit
=
- `duct_map` turns the SubGroup's string count into its ducts, with at most 3 strings per duct. For example, 5 strings becomes `-3-2-`: one 3-string duct and one 2-string duct.
  - Counts above 20 get `NaN`, and the duct length calculation fails later.
- `Number of Ducts` = the number of ducts in the definition. It isn't used later.
- Merges the inverter onto each SubGroup by `GRP`. This is an inner join, so unmatched SubGroups are dropped silently. It also resets the index.
- `N or S or M` compares the inverter Y with the tracker ends (`Y-N-TRK`, `Y-S-TRK`):
  - `S` if the south end is closer,
  - otherwise `M` if the inverter lies between the two ends,
  - otherwise `N`.
  - Because `S` is tested first, an inverter between the ends but nearer the south end becomes `S`, not `M`.
- `closest_y` = where the trench leaves the tracker. It is whichever is closest to the inverter Y: the tracker's north end, the SubGroup's top string midpoint (`Y`), or the tracker's south end.
- Sorted by `GRP`, then `Line`.

In [11]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['N or S or M'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) > abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
    "S",
    np.where(
        (groups_df['inverter_y'] < groups_df['Y-N-TRK']) & (groups_df['inverter_y'] > groups_df['Y-S-TRK']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
groups_df['closest_y'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) < abs(groups_df['Y'] - groups_df['inverter_y']),
    groups_df['Y-N-TRK'],
    np.where(
        abs(groups_df['Y'] - groups_df['inverter_y']) < abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
        groups_df['Y'],
        groups_df['Y-S-TRK']
    )
)

groups_df.sort_values(['GRP', 'Line'], inplace = True)

print(groups_df.head(20))


       GRP             X          Y    Y-N-STR    Y-S-STR    Y-N-TRK  \
0   5.1.01  512930.53125  8085918.5  8085934.5  8085835.0  8085934.5   
1   5.1.01  512930.53125  8085814.0  8085830.0  8085730.0  8085830.0   
5   5.1.01  512937.34375  8085918.5  8085934.5  8085835.0  8085934.5   
6   5.1.01  512937.34375  8085814.0  8085830.0  8085730.0  8085830.0   
10  5.1.01  512944.12500  8085918.5  8085934.5  8085835.0  8085934.5   
11  5.1.01  512944.12500  8085814.0  8085830.0  8085730.0  8085830.0   
15  5.1.01  512950.93750  8085918.5  8085934.5  8085868.5  8085934.5   
16  5.1.02  512950.93750  8085851.0  8085867.0  8085835.0  8085934.5   
17  5.1.02  512950.93750  8085814.0  8085830.0  8085730.0  8085830.0   
22  5.1.02  512957.71875  8085918.5  8085934.5  8085835.0  8085934.5   
23  5.1.02  512957.71875  8085814.0  8085830.0  8085730.0  8085830.0   
27  5.1.02  512964.53125  8085918.5  8085934.5  8085835.0  8085934.5   
28  5.1.02  512964.53125  8085814.0  8085830.0  8085730.0  80858

Breaking the trenches into segments to indicate how many strings there are in each one.
=

- For each inverter (`GRP`), the SubGroups are split into lines west of the inverter line, on it, and east of it.
- Walking from the farthest line toward the inverter, `STR_CUM_TO_INV` accumulates `No. of Strings`. It is the number of strings that the horizontal trench stretch at that line carries toward the inverter.
- SubGroups on the inverter line only get their own count.
- `max_number_of_strings` (the largest value) sets how many `EMF_vala_CC_<n>C` layers the layered trench script loops over. `value_counts()` shows how many stretches get each count.
- Two SubGroups of the same inverter on the same `Line` (e.g. one above and one below a gap) are added one after the other. They get different counts (e.g. 3 and 6), even though their horizontal stretches overlap.
- Writes `groups_df` to `C:\Users\Usuario\Desktop\teste1.xlsx`.

In [12]:
"""
Adds cumulative sum column where values sum towards the inverter_line, calculated separately for each distinct group/segment.
"""
groups_df['STR_CUM_TO_INV'] = 0

# Group by GRP
# Each GRP represents a distinct physical segment with its own inverter
for grp_name in groups_df['GRP'].unique():
    # Get rows belonging to this GRP
    grp_mask = groups_df['GRP'] == grp_name
    grp_indices = groups_df[grp_mask].index

    if len(grp_indices) == 0:
        continue

    # Get the inverter_line for this GRP (should be consistent within group)
    # Using the first occurrence's inverter_line as the target for this group
    inv_line = groups_df.loc[grp_indices[0], 'inverter_line']

    # Separate rows by position relative to inverter_line
    before_inv = []
    at_inv = []
    after_inv = []

    for idx in grp_indices:
        x_val = groups_df.loc[idx, 'Line']
        if x_val < inv_line:
            before_inv.append(idx)
        elif x_val > inv_line:
            after_inv.append(idx)
        else:  # x_val == inv_line
            at_inv.append(idx)

    # Sort before_inv by X ascending (farthest to closest to inverter)
    before_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'])

    # Sort after_inv by X descending (farthest to closest to inverter)
    after_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'], reverse=True)

    # Calculate cumulative sums for before inverter (left to right)
    cumsum = 0
    for idx in before_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum

    # At inverter line: just the value itself
    for idx in at_inv:
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = groups_df.loc[idx, 'No. of Strings']

    # Calculate cumulative sums for after inverter (right to left)
    cumsum = 0
    for idx in after_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum


# Verify the results
print(groups_df[['GRP', 'X', 'No. of Strings', 'inverter_line', 'STR_CUM_TO_INV']].to_string())
max_number_of_strings = groups_df['STR_CUM_TO_INV'].max()
print(groups_df['STR_CUM_TO_INV'].value_counts())
groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")

        GRP             X  No. of Strings  inverter_line  STR_CUM_TO_INV
0    5.1.01  512930.53125               3       512941.0               3
1    5.1.01  512930.53125               3       512941.0               6
5    5.1.01  512937.34375               3       512941.0               9
6    5.1.01  512937.34375               3       512941.0              12
10   5.1.01  512944.12500               3       512941.0               5
11   5.1.01  512944.12500               3       512941.0               8
15   5.1.01  512950.93750               2       512941.0               2
16   5.1.02  512950.93750               1       512961.0               1
17   5.1.02  512950.93750               3       512961.0               4
22   5.1.02  512957.71875               3       512961.0               7
23   5.1.02  512957.71875               3       512961.0              10
27   5.1.02  512964.53125               3       512961.0               7
28   5.1.02  512964.53125               3       512

Preparing the cables
=
- Drops the duplicated merge columns (`UFV_x/_y`, `PST_x/_y`, `INV_x/_y`) and the group ids. Renames the columns as in `groups_df`. (`layer_x/_y` and `STR_x/_y` from the merge are kept.)
- `N or S or M` per string: the same rules as for the SubGroups, but using the string's own `N exit` / `S exit`. So `M` means the inverter Y falls within this string.
- `closest_y` = whichever is closest to the inverter Y: the tracker's north end, the string's own midpoint `Y`, or the tracker's south end.
- `distance2` = 0.5618 (end pile to first/last module) extends `Y-N-STR` / `Y-S-STR`. Neither the cable length nor the cable script uses these columns any more.
- `exit` = where the cable starts on the string: `N exit` for `N`, `S exit` for `S`, and the string midpoint for `M`.

In [13]:
distance2 = 0.5618 # Distance between the first/last pile and the first/last module

cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group'], inplace=True)
cables_df.rename(columns={'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
cables_df['N or S or M'] = np.where(
    abs(cables_df['N exit'] - cables_df['inverter_y']) > abs(cables_df['S exit'] - cables_df['inverter_y']),
    "S",
    np.where(
        (cables_df['inverter_y'] < cables_df['N exit']) & (cables_df['inverter_y'] > cables_df['S exit']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
cables_df['closest_y'] = np.where(
    abs(cables_df['Y-N-TRK'] - cables_df['inverter_y']) < abs(cables_df['Y'] - cables_df['inverter_y']),
    cables_df['Y-N-TRK'],
    np.where(
        abs(cables_df['Y'] - cables_df['inverter_y']) < abs(cables_df['Y-S-TRK'] - cables_df['inverter_y']),
        cables_df['Y'],
        cables_df['Y-S-TRK']
    )
)


cables_df['Y-N-STR'] = cables_df['Y-N-STR'] + distance2
cables_df['Y-S-STR'] = cables_df['Y-S-STR'] - distance2

cables_df['exit'] = np.where(
    cables_df['N or S or M'] == "N",
    cables_df['N exit'],
    np.where(
        cables_df['N or S or M'] == "S",
        cables_df['S exit'],
        cables_df['Y']
    )
)

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

   layer_x                     name             X          Y     GRP STR_x  \
0  MDS_5.1  String_Tracker_1X174p_3  512930.53125  8085918.5  5.1.01         
1  MDS_5.1  String_Tracker_1X174p_2  512930.53125  8085884.5  5.1.01         
2  MDS_5.1  String_Tracker_1X174p_1  512930.53125  8085851.0  5.1.01         
3  MDS_5.1  String_Tracker_1X174p_1  512930.53125  8085814.0  5.1.01         
4  MDS_5.1  String_Tracker_1X174p_2  512930.53125  8085780.0  5.1.01         

   rotation      Line     N exit     S exit  ...    Y-N-TRK    Y-S-TRK  \
0     180.0  512931.0  8085934.5  8085902.5  ...  8085934.5  8085835.0   
1     180.0  512931.0  8085900.5  8085868.5  ...  8085934.5  8085835.0   
2     180.0  512931.0  8085867.0  8085835.0  ...  8085934.5  8085835.0   
3       0.0  512931.0  8085830.0  8085798.0  ...  8085830.0  8085730.0   
4       0.0  512931.0  8085796.0  8085764.0  ...  8085830.0  8085730.0   

   layer_y    inverter_x inverter_y  STR_y  inverter_line N or S or M  \
0  MDS_5.1  5

Cables, trenches and ducts: routes, lengths and scripts
=
**Offsets** (drawing units):
- `distance3` = 2.6208: horizontal offset of the intermediary point from the tracker line, toward the inverter. The comment calls it vertical, but it is applied on X.
- `distance4` = 0.1825 / `distance5` = 0.675: half the inverter size in X / Y. They make the routes end on the inverter edge and not at its centre.
- `distance6` to `distance9` = 0.5 / 0.45 / 0.4 / 0.35: distances from the inverter to the four stretch points.

**`intermediary_coordinates(df)`** (changes `df` in place):
- Inverter on another line:
  - `intermediary_x_1` = `X ± distance3`, toward the inverter.
  - `intermediary_x_2` to `_5` = `inverter_x ∓ distance6..9`. These are extra vertices on the horizontal run just before the inverter. They're only there as grips for stretching the end in AutoCAD, and don't change the length.
  - `inverter_x` moves `distance4` toward the tracker, so the route enters the inverter from the side.
- Inverter on the same line:
  - All intermediary points = `inverter_x`, a straight vertical run.
  - `inverter_y` moves `distance5`: up for `S`, and down for both `N` and `M`.
- It is applied to `cables_df` here and to `script_df` further down.
- **Re-running this cell shifts `cables_df` by the offsets a second time**, because `cables_df` isn't rebuilt in this cell. Re-run from the string ends step instead.
- `cables_df` is exported to `C:\Users\Usuario\Desktop\teste3333.xlsx` right after the offsets are applied.

**Cables** (`cables_df`, one row per string):
- Red length = |`N exit` − `closest_y`| + the diagonal to `(intermediary_x_1, inverter_y)` + the horizontal run to the inverter edge. The black length is the same, using `S exit`. Both totals go into `lengths_df`.
- The script only draws the **positive** cable, starting at `exit`. The negative cable script is commented out.
- **The lengths don't match the drawn cables:** the drawn cable starts at `exit`, but the red length always starts at `N exit`. On top of that, the black length is counted although no black cable is drawn.
- Writes `Script Cabos Tracker-teste.scr`: `PLINEWID 0`, layer `EMF_cabo_solar_positivo`. Each `_.pline` line ends with a space, which ends the command.

**Trenches** (`script_df`, one row per SubGroup, with `Y` = `closest_y`):
- One polyline per SubGroup: tracker exit, then the intermediary point, then the stretch points, then the inverter edge. The polylines of the same inverter overlap along the horizontal run.
- Writes `Script Valas Tracker-teste.scr` (`PLINEWID 0.5`, layer `EMF_bt_vala`).

**Ducts:**
- `calculate_duct_length` = trench route length × the number of times the digit appears in `Duct Definition`. For example, `-3-3-1-` gives two 3-string ducts and one 1-string duct.
- There is one length column and one total for each duct size, 1 to 4. No entry in this `duct_map` has a 4-string duct, so `Dutos 4` is always 0.
- Writes `Script Dutos Tracker-teste.scr` (`PLINEWID 0.03`). Each duct size goes on its own layer, `EMF_bt_eletroduto_1S` to `_4S`, and a layer with no ducts is skipped.
  - The 3-string ducts can repeat, so their polylines are joined with newlines.
  - The 1-, 2- and 4-string polylines are repeated by string multiplication, with no newline between the copies. This works only because each of these sizes appears at most once per definition.

The `.scr` files are written to `C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\`.

In [14]:
script_df = groups_df[['GRP','No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_y', 'X', 'inverter_x', 'inverter_y','Line', 'inverter_line', 'STR_CUM_TO_INV', 'Y-N-STR', 'Y-S-STR', 'N or S or M']].copy()
script_df.rename(columns={'closest_y': 'Y'}, inplace=True)

distance3 = 2.6208 # Vertical distance between the tracker's pile and the inverter.
distance4 = 0.1825 # Distance between the middle point of the inverter and its edge on the X axis.
distance5 = 0.675 # Distance between the middle point of the inverter and its edge on the Y axis.
distance6 = 0.5 # Distance to make an intermediary point for stretching purposes.
distance7 = 0.45 # Distance to make an intermediary point for stretching purposes.
distance8 = 0.4 # Distance to make an intermediary point for stretching purposes.
distance9 = 0.35 # Distance to make an intermediary point for stretching purposes.

def intermediary_coordinates(df):
    df['intermediary_x_1'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['X'] + distance3,
            df['X'] - distance3
        )
    )

    df['intermediary_x_2'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance6,
            df['inverter_x'] + distance6
        )
    )

    df['intermediary_x_3'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance7,
            df['inverter_x'] + distance7
        )
    )

    df['intermediary_x_4'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance8,
            df['inverter_x'] + distance8
        )
    )

    df['intermediary_x_5'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance9,
            df['inverter_x'] + distance9
        )
    )

    df['inverter_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance4,
            df['inverter_x'] + distance4
        )
    )

    df['inverter_y'] = np.where(
        df['inverter_line'] != df['Line'],
        df['inverter_y'],
        np.where(
            df['N or S or M'] == 'S',
            df['inverter_y'] + distance5,
            df['inverter_y'] - distance5
        )
    )

'''

CABOS

'''

intermediary_coordinates(cables_df)
cables_df.to_excel(r"C:\Users\Usuario\Desktop\teste3333.xlsx")
def calculate_cable_length(row, color):
    try:
        x = row['X']
        y_n_str = row['N exit']
        y_s_str = row['S exit']
        closest_y = row['closest_y']
        int_x_1 = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']

        if color == 'red':
            seg1 = math.sqrt((x - x)**2 + (y_n_str - closest_y)**2)
        else:
            seg1 = math.sqrt((x - x)**2 + (y_s_str - closest_y)**2)
        seg2 = math.sqrt((x - int_x_1)**2 + (closest_y - inv_y)**2)
        seg3 = math.sqrt((int_x_1 - inv_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2 + seg3
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

cables_df['Red Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'red'), axis=1)
cables_df['Black Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'black'), axis=1)

total_red_length = cables_df['Red Cable Length'].sum()
total_black_length = cables_df['Black Cable Length'].sum()

total_cable_length = total_red_length + total_black_length

cables_df['Script Red Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
# cables_df['Script Black Cables'] = cables_df.apply(
#     lambda row: f"_.pline {row['X']},{row['exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
#     axis=1
# )

with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\Script Cabos Tracker-teste.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Red Cables"].astype(str)) + '\n')
    # f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    # f.write('\n'.join(cables_df["Script Black Cables"].astype(str)) + '\n')

'''

VALAS

'''

intermediary_coordinates(script_df)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\Script Valas Tracker-teste.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_bt_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''

DUTOS

'''

def calculate_duct_length(row, no_of_strings):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']
        number_of_ducts = row['Duct Definition'].count(no_of_strings)

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return (seg1 + seg2) * number_of_ducts
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Dutos 1 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '1'), axis=1)
script_df['Dutos 2 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '2'), axis=1)
script_df['Dutos 3 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '3'), axis=1)
script_df['Dutos 4 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '4'), axis=1)

total_1str_duct_length = script_df['Dutos 1 Length'].sum()
total_2str_duct_length = script_df['Dutos 2 Length'].sum()
total_3str_duct_length = script_df['Dutos 3 Length'].sum()
total_4str_duct_length = script_df['Dutos 4 Length'].sum()

total_duct_length = total_1str_duct_length + total_2str_duct_length + total_3str_duct_length + total_4str_duct_length

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: '\n'.join([
        f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} "
        for _ in range(str(row['Duct Definition']).count('3'))
    ]),
    axis=1
)
script_df['Script Dutos 4'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('4'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\Script Dutos Tracker-teste.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write('\n'.join(layer3_commands) + '\n')

    # Layer 4
    layer4_commands = script_df["Script Dutos 4"].astype(str)
    layer4_commands = layer4_commands[layer4_commands != '']
    if len(layer4_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_4S\n\n")
        f.write('\n'.join(layer4_commands) + '\n')

0      _.pline 512930.53125,8085835.0 512933.15625,80...
1      _.pline 512930.53125,8085830.0 512933.15625,80...
5      _.pline 512937.34375,8085835.0 512939.96875,80...
6      _.pline 512937.34375,8085830.0 512939.96875,80...
10     _.pline 512944.125,8085835.0 512941.5,8085832....
                             ...                        
246    _.pline 513256.9375,8086438.5 513254.3125,8086...
247    _.pline 513256.9375,8086434.5 513254.3125,8086...
251    _.pline 513263.71875,8086438.5 513261.09375,80...
252    _.pline 513263.71875,8086434.5 513261.09375,80...
256    _.pline 513270.53125,8086438.5 513267.90625,80...
Name: Script Valas, Length: 392, dtype: str


Layered trenches, the lengths summary and the Excel exports
=
Splits each SubGroup's trench into two pieces, so each piece can go on a layer that shows how many strings it carries:
- **Piece 1:** from the tracker exit to `intermediary_x_1`, on layer `EMF_vala_CC_<No. of Strings>C`.
- **Piece 2:** horizontal, from `intermediary_x_1` toward the inverter, one tracker `pitch` long (6.8, the spacing between tracker lines) and capped at the inverter edge (`final_x`, via `intermediary_x_6`). It goes on layer `EMF_vala_CC_<STR_CUM_TO_INV>C`.
  - Chained line by line, these pieces make up the trench to the inverter, and each piece is labelled with its accumulated string count.
  - On the inverter line, piece 1 is the vertical run to the inverter and piece 2 has zero length.
- `pitch` must match the project's tracker line spacing.

Outputs:
- `Valas Length` / `total_trench_length`: the sum of both pieces of every SubGroup.
- `Script Valas Layered Tracker-teste.scr` (`PLINEWID 0.5`): one `LAYER S EMF_vala_CC_<n>C` block for each n from 1 to `max_number_of_strings` that has pieces.
- `lengths_df` (red/black/total cable, duct per size, total duct, total trench) → `C:\Users\Usuario\Desktop\Lengths.xlsx`.
- `script_df` → `SCRIPTS.xlsx` and `cables_df` → `cables.xlsx`, both on the Desktop.
- All the Tracker DC notebooks write to these same Desktop paths, so each run overwrites the previous one.

Finally, the DuckDB connection is closed.

In [15]:
'''

LAYERED VALAS

'''

pitch = 6.8

script_df['intermediary_x_6'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df['intermediary_x_1'] + pitch,
        script_df['intermediary_x_1'] - pitch
    )
)

script_df['final_x'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df.apply(lambda row: min(row['intermediary_x_6'], row['inverter_x']), axis=1),
        script_df.apply(lambda row: max(row['intermediary_x_6'], row['inverter_x']), axis=1)
    )
)

def calculate_trench_length(row):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        final_x = row['final_x']

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - final_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Valas Length'] = script_df.apply(calculate_trench_length, axis=1)

total_trench_length = script_df['Valas Length'].sum()

script_df['Script Valas Layered 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} ",
    axis=1
)

script_df['Script Valas Layered 2'] = script_df.apply(
    lambda row: f"_.pline {row['intermediary_x_1']},{row['inverter_y']} {row['final_x']},{row['inverter_y']} ",
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Dutos DC\minas\Script Valas Layered Tracker-teste.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    for i in range(1, max_number_of_strings+1):
        mask1 = script_df['No. of Strings'] == i
        mask2 = script_df['STR_CUM_TO_INV'] == i

        if mask1.any() or mask2.any():
            f.write(f"LAYER S EMF_vala_CC_{i}C\n\n")

            if mask1.any():
                f.write('\n'.join(script_df[mask1]["Script Valas Layered 1"].astype(str)) + '\n')

            if mask2.any():
                f.write('\n'.join(script_df[mask2]["Script Valas Layered 2"].astype(str)) + '\n')


lengths_df = pd.DataFrame.from_dict(data={
    'DC Red Cable Length': [total_red_length],
    'DC Black Cable Length': [total_black_length],
    'DC Total Cable Length': [total_cable_length],
    '1str Duct Length': [total_1str_duct_length],
    '2str Duct Length': [total_2str_duct_length],
    '3str Duct Length': [total_3str_duct_length],
    '4str Duct Length': [total_4str_duct_length],
    'Total Duct Length': [total_duct_length],
    'Total Trench Length': [total_trench_length]
}, orient="index", columns=['Length (m)'])
lengths_df.to_excel(r"C:\Users\Usuario\Desktop\Lengths.xlsx")
script_df.to_excel(r"C:\Users\Usuario\Desktop\SCRIPTS.xlsx")
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")

conn.close()